In [1]:
!nvidia-smi

Wed Apr 15 16:14:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install tensorflow

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls

drive  sample_data


In [6]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
import os

In [25]:

# Path to the 'maize_dataset' folder
DATA_DIR = '/content/drive/MyDrive/GROW_SAFE/maize_dataset/train'
!ls -a {DATA_DIR}

bad  good


In [36]:

# Image parameters
IMG_DIM = 128
CHANNELS = 3
IMG_SIZE = (IMG_DIM, IMG_DIM, CHANNELS)
BATCH_SIZE = 32

In [37]:
datagen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values
    validation_split=0.2 # Use 20% of data for validation
)

In [38]:
# Load the training data
train_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary', # We have two classes: 'good' and 'bad'
    subset='training'
)

Found 23136 images belonging to 2 classes.


In [39]:
# Load the validation data
validation_generator = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 5784 images belonging to 2 classes.


In [40]:
input_tensor = Input(shape=IMG_SIZE)

# Load the MobileNetV2 base model
# 'weights=imagenet' uses pre-trained weights on a massive dataset
# 'include_top=False' removes the final classification layer, which we will replace

base_model = MobileNetV2(
    input_tensor=input_tensor,
    include_top=False,
    weights='imagenet'
)

/tmp/ipykernel_912/260661250.py:7: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


In [41]:
# Freeze the base model layers
# Prevents the pre-trained weights from being updated during training
base_model.trainable = False

In [42]:
# Build the classification head for our maize problem
x = base_model.output

# Downsample the features
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)

# Sigmoid for binary classification (0 or 1)
output = Dense(1, activation='sigmoid')(x) 

In [43]:

# Create the final model
model = Model(inputs=base_model.input, outputs=output)

# Compile the model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 64, 64,    │        864 │ input_layer_7[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 64, 64,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 64, 64,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 64, 64,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 64, 64,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 64, 64,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 65, 65,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 32, 32,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 32, 32,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Define the path to save the checkpoints
CHECKPOINT_PATH = '/content/drive/MyDrive/GROW_SAFE/v2/maize_classifier_epoch{epoch:02d}_model.h5'

# Create a ModelCheckpoint callback
# Saves the model after each epoch
checkpoint_callback = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    save_weights_only=False, # Set to True if you only want to save weights
    save_best_only=False, # Set to True to save only the best model based on a metric (e.g., 'val_accuracy')
    monitor='val_accuracy', # Metric to monitor if save_best_only is True
    verbose=1
)

# Train the model
# You can increase the number of epochs for better accuracy
EPOCHS = 1

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=[checkpoint_callback] # Add the callback here
)

# Save the model in the Keras SavedModel format (recommended)
# This will save it to your Google Drive
# MODEL_SAVE_PATH = '/content/drive/MyDrive/GROW_SAFE/v2/maize_classifier_model'
# model.save(MODEL_SAVE_PATH)

# print(f"Model saved to: {MODEL_SAVE_PATH}")

In [ ]:
from tensorflow.keras.callbacks import Callback

class AccurateCheckpoint(Callback):
    def __init__(self, checkpoint_path, initial_threshold=0.60, improvement_threshold=0.02):
        super(AccurateCheckpoint, self).__init__()
        self.checkpoint_path = checkpoint_path
        self.initial_threshold = initial_threshold
        self.improvement_threshold = improvement_threshold
        self.best_accuracy = 0

    def on_epoch_end(self, epoch, logs=None):
        current_accuracy = logs.get('accuracy')
        if current_accuracy is None:
            return

        # Check if the accuracy has reached the initial threshold
        if current_accuracy >= self.initial_threshold:
            # Check if the current accuracy is a significant improvement over the best accuracy
            if current_accuracy >= self.best_accuracy + self.improvement_threshold:
                self.best_accuracy = current_accuracy
                filepath = self.checkpoint_path.format(epoch=epoch+1)
                self.model.save(filepath)
                print(f"\nEpoch {epoch+1}: Training accuracy improved to {current_accuracy:.4f}, saving model to {filepath}")
            else:
                print(f"\nEpoch {epoch+1}: Training accuracy {current_accuracy:.4f}, no significant improvement.")
        else:
            print(f"\nEpoch {epoch+1}: Training accuracy {current_accuracy:.4f}, below initial threshold.")

# Instantiate the custom callback
custom_checkpoint_callback = AccurateCheckpoint(
    checkpoint_path='/content/drive/MyDrive/GROW_SAFE/v2/maize_classifier_epoch{epoch:02d}_model.h5',
    initial_threshold=0.60,
    improvement_threshold=0.02
)

# Train the model with the custom callback
EPOCHS = 10 # You might want to increase epochs for better results

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    steps_per_epoch=train_generator.samples // BATCH_SIZE,
    validation_steps=validation_generator.samples // BATCH_SIZE,
    callbacks=[custom_checkpoint_callback] # Use the custom callback here
)

In [ ]:
MODEL_SAVE_PATH = '/content/drive/MyDrive/SAFE_GROW/v2/maize_classifier_epoch1_model.h5'
model.save(MODEL_SAVE_PATH)
print(f"Model successfully saved to: {MODEL_SAVE_PATH}")

In [ ]:
!pip install tensorflowjs

In [ ]:
!tensorflowjs_converter --input_format keras /content/drive/MyDrive/maize_dataset/maize_classifier_epoch1_model.keras /content/drive/MyDrive/maize_dataset/tfjs_model_files

In [ ]:
!tensorflowjs_converter --version